In [ ]:
from __future__ import annotations

import base64
import hashlib

EXPECTED_RUNTIME_SHA256 = 'bde93ca8b684640d6c8baccbd7782cdb627e27449dce39597b42d0828f3ed34f'
RUNTIME_SOURCE_B64 = (
    '...'
    'YWluKCkpCg=='
)

runtime_source = base64.b64decode(RUNTIME_SOURCE_B64).decode('utf-8')
observed = hashlib.sha256(runtime_source.encode('utf-8')).hexdigest()
if observed != EXPECTED_RUNTIME_SHA256:
    raise RuntimeError('runtime source identity mismatch')
namespace = {'__name__': '__main__', 'EXECUTED_RUNTIME_SCRIPT_SHA256': observed}
exec(compile(runtime_source, '<auragateway-p4-output-contract-diagnostic-v2>', 'exec'), namespace)
